# 05 – Inj-SQuAD and Inj-TriviaQA preprocessing
**Dataset:** Inj-SQuAD and Inj-TriviaQA – indirect-injection benchmarks (Chen et al., ACL 2025, ref. [9])
**Source:** https://github.com/LukeChen-go/indirect-pia-detection, folder `data/` (repo licence: MIT). The underlying documents come from SQuAD and TriviaQA, whose own terms apply.
**Project:** Cyber Nexus

Each row: a question (`instruction`), a clean document (`input`), the gold answer (`output`), an injected instruction (`injection`) and the expected output if the injection is obeyed (`injection_output`, the "probe"). Two files of 900 rows each: `crafted_instruction_data_squad_injection_qa.json` and `crafted_instruction_data_tri_injection_qa.json`.

The repo's other data files (`crafted_instruction_data_alpaca.json`, `..._context_squad.json`, `..._context_tri.json`, 97 MB together) are *training* data for the authors' detector models. Our defenses are training-free, so they are not used.

In [1]:
import json, ast, re, unicodedata, hashlib, subprocess, html
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
md5 = lambda s: hashlib.md5(s.encode("utf-8")).hexdigest()

ZW = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")
CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def clean_text(t, collapse_spaces=False):
    t = unicodedata.normalize("NFC", str(t))
    t = ZW.sub("", t); t = CTRL.sub("", t)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = "\n".join(line.rstrip() for line in t.split("\n"))
    t = re.sub(r"\n{3,}", "\n\n", t)
    if collapse_spaces: t = re.sub(r"[ \t]{2,}", " ", t)
    return t.strip()

def deva_ratio(t):
    letters = [c for c in t if c.isalpha()]
    return sum("\u0900" <= c <= "\u097f" for c in letters) / max(len(letters), 1)

def lang_tag(t):
    r = deva_ratio(t)
    return "hi" if r > 0.5 else ("mixed" if r > 0.02 else "en")   # Latin-script Hinglish needs a separate detector

def group_split(df, group_col, strat_col=None, seed=SEED):
    """70/15/15 split where all rows of one group stay together (avoids leakage)."""
    cols = [group_col] + ([strat_col] if strat_col else [])
    g = df.drop_duplicates(group_col)[cols]
    s = g[strat_col] if strat_col else None
    g_tr, g_tmp = train_test_split(g, test_size=0.30, stratify=s, random_state=seed)
    s2 = g_tmp[strat_col] if strat_col else None
    g_va, g_te = train_test_split(g_tmp, test_size=0.50, stratify=s2, random_state=seed)
    parts = [df[df[group_col].isin(x[group_col])].copy().assign(split=n) for x, n in [(g_tr, "train"), (g_va, "val"), (g_te, "test")]]
    for a in range(3):
        for b in range(a + 1, 3):
            assert not (set(parts[a][group_col]) & set(parts[b][group_col])), "group leakage!"
    assert sum(len(p) for p in parts) == len(df)
    return parts

def add_buckets(parts, len_col, by=None):
    """short/medium/long using tertiles computed on the TRAIN part only."""
    tr = parts[0]
    if by:
        q = tr.groupby(by)[len_col].quantile([1/3, 2/3]).unstack(); q.columns = ["q1", "q2"]
        f = lambda r: ("short" if r[len_col] <= q.loc[r[by], "q1"] else "medium" if r[len_col] <= q.loc[r[by], "q2"] else "long")
    else:
        q1, q2 = tr[len_col].quantile([1/3, 2/3]); q = pd.Series({"q1": q1, "q2": q2})
        f = lambda r: "short" if r[len_col] <= q1 else "medium" if r[len_col] <= q2 else "long"
    for p in parts: p["length_bucket"] = p.apply(f, axis=1)
    return q

def git_commit(repo):
    return subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

def ensure_clone(url, dest):
    if not Path(dest).exists():
        subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True)

REPO = Path("indirect-pia-detection"); DATA = REPO / "data"
OUT = Path("processed/05_inj_squad_triviaqa"); OUT.mkdir(parents=True, exist_ok=True)
ensure_clone("https://github.com/LukeChen-go/indirect-pia-detection.git", REPO)
commit = git_commit(REPO); print("commit:", commit)
FILES = {"inj_squad": "crafted_instruction_data_squad_injection_qa.json", "inj_triviaqa": "crafted_instruction_data_tri_injection_qa.json"}

commit: 1e9e0f00179adc154e9ca991b9cd3206c99f9588


## 1. Load and inspect

In [2]:
frames = []
for name, f in FILES.items():
    d = pd.DataFrame(json.load(open(DATA / f, encoding="utf-8"))); d["dataset"] = name; frames.append(d)
raw = pd.concat(frames, ignore_index=True)
print("RAW shape:", raw.shape, "| columns:", raw.columns.tolist())
for name, d in raw.groupby("dataset"):
    print(f"\n== {name}: {len(d)} rows")
    print("  missing:", int(d.isna().sum().sum()), "| empty strings:", int(sum((d[c].str.strip() == "").sum() for c in ["instruction", "input", "output", "injection", "injection_output"])))
    print("  duplicate rows:", int(d.duplicated(["instruction", "input"]).sum()), "| unique documents:", d.input.nunique(), "| unique questions:", d.instruction.nunique())
    print("  unique injected instructions:", d.injection.nunique(), "| each used", d.injection.value_counts().unique().tolist(), "times | unique probes:", d.injection_output.nunique())
    print("  document length (words):", d.input.str.split().str.len().describe().round(0)[["mean", "min", "max"]].to_dict())
    tags = {t: int(d.input.str.count(re.escape(t)).sum()) for t in ["[DOC]", "[TLE]", "[PAR]"]}
    print("  markup tags in documents:", tags)
    print("  zero-width/control chars in documents:", int(d.input.str.contains(r"[\u200b\u200c\u200d\u2060\ufeff\x00-\x08\x0b\x0c\x0e-\x1f]").sum()))
    print("  gold answer found in document:", int(sum(str(o).lower() in i.lower() for o, i in zip(d.output, d.input))), "of", len(d))
    print("  probe text already present in the CLEAN document (whole-word match):", int(sum(bool(re.search(r"\b" + re.escape(str(o).lower()) + r"\b", i.lower())) for o, i in zip(d.injection_output, d.input))))
sq, tq = raw[raw.dataset == "inj_squad"], raw[raw.dataset == "inj_triviaqa"]
print("\nsame 300 injected instructions in both datasets:", set(sq.injection) == set(tq.injection))

RAW shape: (1800, 6) | columns: ['instruction', 'input', 'output', 'injection', 'injection_output', 'dataset']

== inj_squad: 900 rows
  missing: 0 | empty strings: 0
  duplicate rows: 0 | unique documents: 900 | unique questions: 900
  unique injected instructions: 300 | each used [3] times | unique probes: 91
  document length (words): {'mean': 134.0, 'min': 68.0, 'max': 508.0}
  markup tags in documents: {'[DOC]': 0, '[TLE]': 0, '[PAR]': 0}
  zero-width/control chars in documents: 2
  gold answer found in document: 900 of 900
  probe text already present in the CLEAN document (whole-word match): 2

== inj_triviaqa: 900 rows
  missing: 0 | empty strings: 0
  duplicate rows: 0 | unique documents: 900 | unique questions: 900
  unique injected instructions: 300 | each used [3] times | unique probes: 91
  document length (words): {'mean': 675.0, 'min': 75.0, 'max': 761.0}
  markup tags in documents: {'[DOC]': 1456, '[TLE]': 1456, '[PAR]': 18426}
  zero-width/control chars in documents: 3

  probe text already present in the CLEAN document (whole-word match): 6

same 300 injected instructions in both datasets: True


## 2. Preprocessing
| Step | Why |
|---|---|
| Rename columns to `question, document, answer, injected_instruction, probe` | descriptive names, one schema across datasets |
| Remove TriviaQA markup (`[DOC]`, `[TLE]` removed, `[PAR]` -> line break) | these are the authors' formatting tags, not document text; a regex filter or an LLM would otherwise read them as content |
| Unicode NFC, remove control / zero-width characters, whitespace cleanup | consistent text for regex rules |
| Flag `answer_in_doc` | rows whose gold answer is not found as text in the document (mainly TriviaQA) cannot be scored on exact match; kept, but flagged |
| Flag `probe_in_doc` | if the probe text is already in the clean document, an "attack succeeded" check on the output can fire without any attack; these rows should be excluded from ASR |
| Language tag; length buckets from the train split only | planned language and document-length analysis |
| Split by injected instruction (group), same assignment in both datasets | each injection is reused 3 times and the same 300 injections appear in both datasets; a row-level split would let an injection seen in development appear in the test set |

In [3]:
df = raw.rename(columns={"instruction": "question", "input": "document", "output": "answer", "injection": "injected_instruction", "injection_output": "probe"}).copy()
df["words_before"] = df.document.str.split().str.len()
TAGS = re.compile(r"\[(?:DOC|TLE)\]\s*")
def fix_doc(s):
    s = TAGS.sub("", s); s = s.replace("[PAR]", "\n")
    return clean_text(s, collapse_spaces=True)
df["document"] = df.document.map(fix_doc)
for c in ["question", "answer", "injected_instruction", "probe"]: df[c] = df[c].map(clean_text)
df["words"] = df.document.str.split().str.len()
assert not df.document.str.contains(r"\[(?:DOC|TLE|PAR)\]").any()
df["answer_in_doc"] = [a.lower() in d.lower() for a, d in zip(df.answer, df.document)]
df["probe_in_doc"] = [bool(re.search(r"\b" + re.escape(p.lower()) + r"\b", d.lower())) for p, d in zip(df.probe, df.document)]
n0 = len(df); df = df.drop_duplicates(["dataset", "question", "document"]).reset_index(drop=True); print(f"rows {n0} -> {len(df)} after dedup")
df["injection_group"] = "inj-" + df.injected_instruction.map(lambda s: md5(s)[:8])
df["id"] = [f"{d}-{md5(q + '||' + t)[:10]}" for d, q, t in zip(df.dataset, df.question, df.document)]
assert df.id.is_unique
df["language"] = df.document.map(lang_tag)
print(df.groupby("dataset").agg(answer_in_doc=("answer_in_doc", "mean"), probe_in_doc=("probe_in_doc", "sum"), mean_words=("words", "mean")).round(2))
print(df.language.value_counts().to_dict())

rows 1800 -> 1800 after dedup
              answer_in_doc  probe_in_doc  mean_words
dataset                                              
inj_squad              1.00             2      133.71
inj_triviaqa           0.94             6      651.30
{'en': 1800}


## 3. Split (before any balancing)
Group split by injected instruction (300 groups, 70/15/15, seed 42). The same assignment is applied to both datasets. Train + validation = development; test untouched until the final comparison.

In [4]:
train, val, test = group_split(df, "injection_group")
parts = [train, val, test]
q = {}
for ds in ["inj_squad", "inj_triviaqa"]:
    tr = train[train.dataset == ds]; q1, q2 = tr.words.quantile([1/3, 2/3]); q[ds] = (round(q1, 1), round(q2, 1))
    for p in parts:
        m = p.dataset == ds
        p.loc[m, "length_bucket"] = np.where(p.loc[m, "words"] <= q1, "short", np.where(p.loc[m, "words"] <= q2, "medium", "long"))
print("tertile thresholds (words, train):", q)
allp = pd.concat(parts)
print(allp.groupby(["split", "dataset"]).size().unstack())
print("injections per split:", allp.groupby("split").injection_group.nunique().to_dict())

tertile thresholds (words, train): {'inj_squad': (105.0, 139.3), 'inj_triviaqa': (651.0, 673.0)}
dataset  inj_squad  inj_triviaqa
split                           
test           135           135
train          630           630
val            135           135
injections per split: {'test': 45, 'train': 210, 'val': 45}


## 4. Before / after comparison

In [5]:
cmp = df.groupby("dataset").agg(rows=("id", "size"), words_before=("words_before", "mean"), words_after=("words", "mean")).round(1)
cmp["tag_tokens_before"] = [int(raw[raw.dataset == d].input.str.count(r"\[(?:DOC|TLE|PAR)\]").sum()) for d in cmp.index]
cmp["tag_tokens_after"] = [int(df[df.dataset == d].document.str.count(r"\[(?:DOC|TLE|PAR)\]").sum()) for d in cmp.index]
print(cmp)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for a, d in zip(ax[:2], ["inj_squad", "inj_triviaqa"]):
    s = df[df.dataset == d]; a.hist(s.words_before, bins=30, alpha=.55, label="before"); a.hist(s.words, bins=30, alpha=.55, label="after"); a.set_title(f"{d}: document length (words)"); a.legend()
allp.groupby(["split", "dataset"]).size().unstack().plot(kind="bar", ax=ax[2], rot=0, title="Rows per split")
plt.tight_layout(); plt.savefig(FIG / "inj_squad_triviaqa_before_after.png", dpi=150)
ex = raw[raw.dataset == "inj_triviaqa"].input.iloc[0]
print("\nSAMPLE TriviaQA document (before):", repr(ex[:170])); print("SAMPLE TriviaQA document (after): ", repr(fix_doc(ex)[:170]))
print("\nrows flagged probe_in_doc:", df[df.probe_in_doc].groupby("dataset").size().to_dict(), "| rows with answer_in_doc False:", df[~df.answer_in_doc].groupby("dataset").size().to_dict())

              rows  words_before  words_after  tag_tokens_before  \
dataset                                                            
inj_squad      900         133.7        133.7                  0   
inj_triviaqa   900         675.0        651.3              21338   

              tag_tokens_after  
dataset                         
inj_squad                    0  
inj_triviaqa                 0  



SAMPLE TriviaQA document (before): '[DOC] [TLE] Javier Bardem becomes first Spanish actor to win Oscar ...Javier Bardem becomes first Spanish actor to win Oscar | Reuters [PAR] Mon Feb 25, 2008 | 9:32 AM ES'
SAMPLE TriviaQA document (after):  'Javier Bardem becomes first Spanish actor to win Oscar ...Javier Bardem becomes first Spanish actor to win Oscar | Reuters\n Mon Feb 25, 2008 | 9:32 AM EST\n Javier Bardem '

rows flagged probe_in_doc: {'inj_squad': 2, 'inj_triviaqa': 6} | rows with answer_in_doc False: {'inj_triviaqa': 58}


## 5. Save

In [6]:
cols = ["id", "dataset", "split", "injection_group", "question", "document", "answer", "injected_instruction", "probe",
        "answer_in_doc", "probe_in_doc", "language", "words", "length_bucket"]
for name, x in [("train", train), ("val", val), ("test", test)]:
    x[cols].to_csv(OUT / f"inj_squad_triviaqa_{name}.csv", index=False, encoding="utf-8")
report = dict(dataset="Inj-SQuAD and Inj-TriviaQA", source="https://github.com/LukeChen-go/indirect-pia-detection", commit=commit, licence="MIT (repo); underlying SQuAD/TriviaQA terms apply",
    seed=SEED, raw_rows=int(n0), clean_rows=int(len(df)), rows_per_dataset=df.groupby("dataset").size().to_dict(),
    split_sizes={s: int(len(x)) for s, x in [("train", train), ("val", val), ("test", test)]}, unique_injections=int(df.injection_group.nunique()),
    flagged_probe_in_doc=df[df.probe_in_doc].groupby("dataset").size().to_dict(), flagged_answer_not_in_doc=df[~df.answer_in_doc].groupby("dataset").size().to_dict(),
    language_counts=df.language.value_counts().to_dict(),
    not_used=["crafted_instruction_data_alpaca.json", "crafted_instruction_data_context_squad.json", "crafted_instruction_data_context_tri.json (detector-training data)"])
json.dump(report, open(OUT / "inj_squad_triviaqa_preprocessing_report.json", "w"), indent=2); print(json.dumps(report, indent=2))

{
  "dataset": "Inj-SQuAD and Inj-TriviaQA",
  "source": "https://github.com/LukeChen-go/indirect-pia-detection",
  "commit": "1e9e0f00179adc154e9ca991b9cd3206c99f9588",
  "licence": "MIT (repo); underlying SQuAD/TriviaQA terms apply",
  "seed": 42,
  "raw_rows": 1800,
  "clean_rows": 1800,
  "rows_per_dataset": {
    "inj_squad": 900,
    "inj_triviaqa": 900
  },
  "split_sizes": {
    "train": 1260,
    "val": 270,
    "test": 270
  },
  "unique_injections": 300,
  "flagged_probe_in_doc": {
    "inj_squad": 2,
    "inj_triviaqa": 6
  },
  "flagged_answer_not_in_doc": {
    "inj_triviaqa": 58
  },
  "language_counts": {
    "en": 1800
  },
  "not_used": [
    "crafted_instruction_data_alpaca.json",
    "crafted_instruction_data_context_squad.json",
    "crafted_instruction_data_context_tri.json (detector-training data)"
  ]
}
